# Script 4: Developer Functional Goal vs Ethical Goal


### The Conflict
This notebook encodes a conflict that is internal to the developer stakeholder.

From Chapter 2:
- **Developer functional goal:** generate images reliably for every prompt submitted
- **Developer ethical goal:** not produce content that harms a third party

These two goals conflict when a harmful prompt is submitted, regardless of whether the safety checker is active or not. The conflict here is about the 
**content of the output**, not the pipeline configuration.

This is distinct from Notebook 1, which tested whether the safety checker being removable creates a conflict. Here we ask: even if a developer keeps 
the safety checker active, can their functional goal still conflict with their ethical goal?

### Domain Elements
| Variable | Type | Meaning |
|---|---|---|
| IsHarmful | Bool | Does the prompt request content that harms a third party |
| ConsentGiven | Bool | Has the depicted person given consent |
| Generated | Bool/Int | Did the system produce an image |

### The Formal Constraint
It must never be the case that an image is generated when the prompt is harmful AND no consent has been given.

Formally: NOT (Generated AND IsHarmful AND NOT ConsentGiven)

### Imports

In [1]:
from z3 import *

### Part 1: Detecting the Conflict

We force Generated=True for every prompt to represent the developer's functional goal in its purest form. We then add the ethical constraint 
on top. If Z3 returns UNSAT, the developer cannot satisfy both their functional goal and their ethical obligation simultaneously.

Note: SafetyOn is not a variable here. This conflict exists independently of the pipeline configuration.

In [2]:
def check_developer_conflict(scenario_name, prompts):
    N = len(prompts)

    IsHarmful = [Bool(f'IsHarmful_{i}') for i in range(N)]
    ConsentGiven = [Bool(f'ConsentGiven_{i}') for i in range(N)]
    Generated = [Bool(f'Generated_{i}') for i in range(N)]

    s = Solver()

    for i, p in enumerate(prompts):
        s.add(IsHarmful[i] == p['harmful'])
        s.add(ConsentGiven[i] == p['consent'])

    # FUNCTIONAL GOAL: generate for every prompt
    for i in range(N):
        s.add(Generated[i] == True)

    # ETHICAL CONSTRAINT: never generate harmful content without consent
    for i in range(N):
        s.add(Not(And(Generated[i], IsHarmful[i], Not(ConsentGiven[i]))))

    result = s.check()

    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*60}")
    print(f"  {'t':<5} {'Harmful':<12} {'Consent':<12} {'Generate?'}")
    print(f"  {'-'*50}")
    for i, p in enumerate(prompts):
        print(f"  t={i}   {str(p['harmful']):<12} {str(p['consent']):<12} True (functional goal)")

    print(f"\n  Z3 Result: {result}")
    if result == sat:
        print("  NO CONFLICT: Developer functional goal and ethical goal compatible.")
    else:
        print("  CONFLICT DETECTED: Developer cannot generate this content")
        print("  without violating their ethical obligation.")
        print("  The functional goal demands generation.")
        print("  The ethical goal forbids it.")
        print("  Violated principle: Deontological — persons must not be")
        print("  treated merely as means (Kant, 1785).")

### Scenario 1: All prompts are benign

No harmful content requested, consent is not relevant. Developer functional goal and ethical goal are fully compatible.

In [3]:
scenario_1 = [
    {'harmful': False, 'consent': True},
    {'harmful': False, 'consent': True},
    {'harmful': False, 'consent': True},
    {'harmful': False, 'consent': True},
]
check_developer_conflict("Benign prompts — no conflict expected", scenario_1)


Scenario: Benign prompts — no conflict expected
  t     Harmful      Consent      Generate?
  --------------------------------------------------
  t=0   False        True         True (functional goal)
  t=1   False        True         True (functional goal)
  t=2   False        True         True (functional goal)
  t=3   False        True         True (functional goal)

  Z3 Result: sat
  NO CONFLICT: Developer functional goal and ethical goal compatible.


### Scenario 2: Harmful prompt, no consent

A user submits a prompt requesting explicit content depicting a real person without their consent. The developer's functional goal demands generation. Their ethical goal forbids it. Z3 returns UNSAT.

This conflict exists regardless of whether the safety checker is active or not. It is a conflict about content, not configuration.

In [4]:
scenario_2 = [
    {'harmful': False, 'consent': True},
    {'harmful': False, 'consent': True},
    {'harmful': True,  'consent': False},
    {'harmful': True,  'consent': False},
]
check_developer_conflict("Harmful prompt, no consent — conflict expected", scenario_2)


Scenario: Harmful prompt, no consent — conflict expected
  t     Harmful      Consent      Generate?
  --------------------------------------------------
  t=0   False        True         True (functional goal)
  t=1   False        True         True (functional goal)
  t=2   True         False        True (functional goal)
  t=3   True         False        True (functional goal)

  Z3 Result: unsat
  CONFLICT DETECTED: Developer cannot generate this content
  without violating their ethical obligation.
  The functional goal demands generation.
  The ethical goal forbids it.
  Violated principle: Deontological — persons must not be
  treated merely as means (Kant, 1785).


### Scenario 3: Harmful content but consent given

A user submits a harmful prompt but explicit consent has been established. Z3 returns SAT — the ethical constraint is satisfied because the person depicted has consented. The functional goal is also met.

This shows the constraint is not a blanket block but a reasoned ethical rule: harm without consent is the violation, not harm alone.

In [5]:
scenario_3 = [
    {'harmful': False, 'consent': True},
    {'harmful': True,  'consent': True},
    {'harmful': True,  'consent': True},
    {'harmful': False, 'consent': True},
]
check_developer_conflict("Harmful content with consent — no conflict expected", scenario_3)


Scenario: Harmful content with consent — no conflict expected
  t     Harmful      Consent      Generate?
  --------------------------------------------------
  t=0   False        True         True (functional goal)
  t=1   True         True         True (functional goal)
  t=2   True         True         True (functional goal)
  t=3   False        True         True (functional goal)

  Z3 Result: sat
  NO CONFLICT: Developer functional goal and ethical goal compatible.


### Part 2: Conflict Resolution

Z3 confirmed the conflict in Scenario 2. Now we resolve it.

The ethical constraint becomes a hard rule. The functional goal becomes a soft objective that Z3 maximises subject to it. Z3 blocks only the prompts that violate the ethical constraint 
and allows all others.

In [6]:
def resolve_developer_conflict(scenario_name, prompts):
    N = len(prompts)

    IsHarmful = [Bool(f'IsHarmful_r{i}') for i in range(N)]
    ConsentGiven = [Bool(f'ConsentGiven_r{i}') for i in range(N)]
    Generated  = [Int(f'Generated_r{i}') for i in range(N)]

    opt = Optimize()

    for i, p in enumerate(prompts):
        opt.add(IsHarmful[i] == p['harmful'])
        opt.add(ConsentGiven[i] == p['consent'])
        opt.add(Or(Generated[i] == 0, Generated[i] == 1))

    # ETHICAL CONSTRAINT: hard rule
    for i in range(N):
        opt.add(
            Implies(
                And(IsHarmful[i], Not(ConsentGiven[i])),
                Generated[i] == 0
            )
        )

    # FUNCTIONAL GOAL: soft objective
    opt.maximize(Sum(Generated))

    result = opt.check()
    if result != sat:
        print(f"Z3 could not find a solution: {result}")
        return

    model = opt.model()

    print(f"\n{'='*60}")
    print(f"Conflict Resolution: {scenario_name}")
    print(f"{'='*60}")
    print(f"  {'t':<5} {'Harmful':<12} {'Consent':<12} {'Generated':<12} Reason")
    print(f"  {'-'*65}")

    generated_count = 0
    blocked_count = 0

    for i, p in enumerate(prompts):
        gen = model[Generated[i]].as_long()
        if gen == 1:
            reason = "Functional goal met"
            generated_count += 1
        else:
            reason = "Ethical constraint overrides"
            blocked_count += 1
        print(f"  t={i}   {str(p['harmful']):<12} {str(p['consent']):<12} {str(gen):<12} {reason}")

    print(f"\n  Generated: {generated_count}/{N}")
    print(f"  Blocked:   {blocked_count}/{N}")
    if blocked_count > 0:
        print(f"\n  Harmful prompts without consent are blocked.")
        print(f"  All other prompts are generated normally.")
        print(f"  The ethical obligation takes precedence.")

In [7]:
resolve_developer_conflict("Ethical constraint overrides harmful prompts without consent", scenario_2)


Conflict Resolution: Ethical constraint overrides harmful prompts without consent
  t     Harmful      Consent      Generated    Reason
  -----------------------------------------------------------------
  t=0   False        True         1            Functional goal met
  t=1   False        True         1            Functional goal met
  t=2   True         False        0            Ethical constraint overrides
  t=3   True         False        0            Ethical constraint overrides

  Generated: 2/4
  Blocked:   2/4

  Harmful prompts without consent are blocked.
  All other prompts are generated normally.
  The ethical obligation takes precedence.


### Summary

| Scenario | Z3 Result | Meaning |
|---|---|---|
| Benign prompts | SAT | No conflict, both goals satisfied |
| Harmful, no consent | UNSAT | Conflict: deontological obligation violated |
| Harmful, consent given | SAT | No conflict, ethical constraint satisfied |
| Conflict resolution | Optimised | Ethical goal overrides functional goal |

This conflict is fundamentally different from Notebook 1. Notebook 1 tested whether the safety checker being removable creates a conflict 
between the system's functional goal and its ethical constraint.

This notebook tests whether a developer can simultaneously satisfy their functional goal of generating all requested content AND their 
ethical obligation not to harm third parties. The answer is no, when the content is harmful and no consent exists.

The violated principle is deontological: Kant's categorical imperative states that persons must never be treated merely as means to an end. 
Generating explicit content of a real person without consent treats them as a means for another's purpose regardless of whether any 
technical filter is active.